In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from multimodal_lancedb import *
from utils import *
from ground_truth import *
from judge import *
import pandas as pd
import lancedb

import PIL.Image
from prompt import *

In [2]:
# Initialize the system
search_system = MusicSearchSystem(db_path="./.lancedb_2", music_dir="music")

In [3]:
api_key = os.getenv('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)
video_path = 'video/'
input = client.files.upload(file=f"{video_path}19. fantasy_video.mp4")

In [37]:
# generate summary for image
response = client.models.generate_content(
    model="gemini-2.0-flash",
    config=types.GenerateContentConfig(
                system_instruction=SYS_SUMMARY_PROMPT_VIDEO,
                temperature=0.7
    ),
    contents=[USER_PROMPT_VIDEO, input]
)

video_summary = response.text

print(video_summary)

This scene evokes a sense of wonder and fantasy, featuring a fairy-like figure with iridescent wings against a twilight seascape. The atmosphere calls for whimsical and ethereal music, perhaps incorporating gentle synth pads and delicate piano melodies. This could suit a fantasy-themed video, a dreamy product ad, or a cinematic scene needing a touch of magic.


In [17]:
db = lancedb.connect("./.lancedb_2")
table_audio = db.open_table("music_audio")
audio_embedding_df = table_audio.to_pandas()

audio_embedding_df

,song_name,song_path,audio_vector
0,My Rhapsody Sounds - Short Version A,music/Assaf Ayalon - My Rhapsody Sounds - Shor...,"[-0.029478367, 0.009709472, 0.05185532, 0.0254..."
1,Laid Back - Short Version A,music/The Mind Sweepers - Laid Back - Short Ve...,"[-0.033787563, 0.040226605, 0.004442984, 0.081..."
2,Far Taj,music/ZISO - Far Taj.mp3,"[-0.018721217, 0.035051364, 0.05141652, 0.0356..."
3,The Stones - Short Version,music/Wild Tulip - The Stones - Short Version.mp3,"[-0.008583562, 0.020294761, 0.0050604204, 0.03..."
4,Fixed - Short Version B,music/Swirling Ship - Fixed - Short Version B.mp3,"[0.0031058518, 0.020443501, 0.0060475464, -0.0..."
...,...,...,...
195,Orchestral News Intro,music/Tomasz_Redman - Orchestral News Intro.mp3,"[0.010141604, -0.017981885, 0.014272054, -0.03..."
196,Upbeat Happy Fun Logo,music/puremusic - Upbeat Happy Fun Logo.mp3,"[-0.031311695, -0.03104818, 0.022266045, 0.022..."
197,Happy Birthday In Paris,music/Music-Ideas - Happy Birthday In Paris.mp3,"[-0.049790498, -0.03626891, 0.059174698, -0.00..."
198,Funny Game Loop,music/honey_lemon - Funny Game Loop.wav,"[-0.045391183, -0.033096816, 0.025908915, -0.0..."


In [18]:
table_text = db.open_table("music_text")
text_embedding_df = table_text.to_pandas()

text_embedding_df

,source,song_name,artist,mood,video_theme,genre,instrument,other_tags,bpm,lmm_description,combined_info,text_vector
0,Artlist,My Rhapsody Sounds - Short Version A,Assaf Ayalon,"Uplifting, Happy, Carefree, Love, Playful","Business, Food, Education, Lifestyle, Urban","Cinematic, Acoustic, Pop, Folk, Children, Corp...","Acoustic Guitar, Keys",,145.0,A positive and uplifting acoustic folk track w...,"Moods: Uplifting, Happy, Carefree, Love, Playf...","[0.00016941165, -0.011131651, -0.004014101, -0..."
1,Artlist,Laid Back - Short Version A,The Mind Sweepers,"Powerful, Serious, Angry","Road Trip, Sport & Fitness, Fashion, Industry",Rock,"Electric, Guitar, Acoustic Drums",,78.0,This is a powerful and energetic rock music tr...,"Moods: Powerful, Serious, Angry. Video Themes:...","[-0.0041819224, -0.01964485, -0.021090291, -0...."
2,Artlist,Far Taj,ZISO,"Uplifting, Powerful, Carefree, Groovy","Travel, Shorts","World, Electronic, Hip Hop","Ethnic, Electronic Drums, Bass",,96.0,A traditional Indian Bhangra track with modern...,"Moods: Uplifting, Powerful, Carefree, Groovy. ...","[-0.011723319, -0.008885955, 0.0040757894, -0...."
3,Artlist,The Stones - Short Version,Wild Tulip,"Love, Serious, Dramatic, Sad, Hopeful","Time-Lapse, Documentary, Road Trip, Medical, L...",Cinematic,Piano,,69.0,This piece is a solo piano instrumental with a...,"Moods: Love, Serious, Dramatic, Sad, Hopeful. ...","[0.0012076573, -0.0034555339, -0.0020917628, -..."
4,Artlist,Fixed - Short Version B,Swirling Ship,"Serious, Dramatic, Scary, Dark","Time-Lapse, Drone Shots, Nature, Slow Motion","Ambient, Country, Cinematic","Electric Guitar, Synth, Electronic Drums, Pads",,121.0,"The music is mysterious and dramatic, featurin...","Moods: Serious, Dramatic, Scary, Dark. Video T...","[-0.0021377725, -0.012590199, -0.013713269, -0..."
...,...,...,...,...,...,...,...,...,...,...,...,...
195,envato,Orchestral News Intro,Tomasz_Redman,"energetic, epic, powerful, solemn, uplifting","announcement, broadcast news, broadcasting, bu...",corporate,strings,global,125.0,This is a dynamic and uplifting music track th...,"Moods: energetic, epic, powerful, solemn, upli...","[-0.006809455, -0.016746698, -0.016968248, -0...."
196,envato,Upbeat Happy Fun Logo,puremusic,"bouncy, bright, catchy, cheerful, energetic, f...","commercial, happy logo, intro, kids, logo, sum...","acoustic, children","claps, ukulele","melody, youth",NaN,"A positive, upbeat, cheerful, and happy acoust...","Moods: bouncy, bright, catchy, cheerful, energ...","[0.002132031, -0.005020746, -0.0029374287, -0...."
197,envato,Happy Birthday In Paris,Music-Ideas,"cheerful, funny, happy, lively, playful, upbeat","ads, advertising, birthday, broadway, casino, ...","bigband, jazz, retro","accordion, piano, trumpets","france, french, paris",120.0,A fun and lively Latin track featuring a varie...,"Moods: cheerful, funny, happy, lively, playful...","[-0.012678004, -0.011919039, -0.00089095806, -..."
198,envato,Funny Game Loop,honey_lemon,"comical, fun, funny, laugh, smile, soft","cartoon, comedy, comic, kids, short, summer, tv","acoustic, children, folk, jazz",bells,loop,170.0,"A casual, jazzy, swing music with vibraphone, ...","Moods: comical, fun, funny, laugh, smile, soft...","[-0.012116052, -0.01674075, 0.010771282, -0.02..."


In [19]:
recommendation_block = ''
recommendation_block += f"{text_embedding_df['artist'][1]} - {text_embedding_df['song_name'][1]}\n"
recommendation_block += f"Information: {text_embedding_df['combined_info'][1]}\n\n"
print(recommendation_block) 

The Mind Sweepers - Laid Back - Short Version A
Information: Moods: Powerful, Serious, Angry. Video Themes: Road Trip, Sport & Fitness, Fashion, Industry. Instruments: Electric, Guitar, Acoustic Drums. Genres: Rock. Other tags: . Description: This is a powerful and energetic rock music track with catchy electric guitar riffs, hard hitting drums, and upbeat bass. The track is perfect for use in sports videos, advertising, commercials, corporate videos, and more. It will certainly add a touch of energy and excitement to your project.




In [4]:
# Search the music from query
query = '''I’m looking for a tool to clear my browsing history.'''
results = search_system.search_music(query, top_k=200, use_top_n_context = 1)

# play music
# print("\nOverlapping Music：")
# for audio_path in results['audio_paths']:
#     print(f"\nNow playing: {os.path.basename(audio_path)}")
#     display(Audio(audio_path))

# Show explanation of LLM
df_recommendations = pd.DataFrame(results["final_results"])
df_recommendations

# print("\nLLM explanation：")
# print(results['explanation'])

[2025-06-13T17:28:00Z WARN  lance::dataset] No existing dataset at /home/tinglin/1125_env/experiment/.lancedb_temp/rerank_tmp.lance, it will be created


,song_name,artist,mood,video_theme,genre,instrument,other_tags,description,similarity_score,similarity_audio,similarity_text,source,rerank_score,audio_path,combined_info
0,Promised - Short Version B,Ziv Moran,"Uplifting, Powerful, Hopeful, Groovy, Exciting","Road Trip, Lifestyle, Urban, Medical, Landscap...","Acoustic, Pop, Folk","Acoustic Guitar, Piano, Acoustic Drums, Bells,...",,"A driving, upbeat, acoustic guitar-based instr...",0.812127,0.659375,0.877592,"audio,text",0.000034,music/Ziv Moran - Promised - Short Version B.mp3,"Moods: Uplifting, Powerful, Hopeful, Groovy, E..."
1,Infinite - Short Version,Kuyani,"Uplifting, Carefree, Hopeful, Groovy","Technology, Time-Lapse, Lifestyle, Urban, Dron...","Electronic, Pop","Synth, Keys, Electronic Drums, Pads",,A cool and groovy synthwave track with a retro...,0.860492,0.764911,0.901456,"audio,text",0.000034,music/Kuyani - Infinite - Short Version.mp3,"Moods: Uplifting, Carefree, Hopeful, Groovy. V..."
2,Shaal Region - Short Version,Swirling Ship,"Carefree, Love, Peaceful, Playful, Hopeful","Business, Food, Education, Documentary, Road T...","Cinematic, Acoustic, Folk","Acoustic Guitar, Piano, Acoustic Drums",,A calm and warm acoustic guitar track that evo...,0.891949,1.000000,0.845642,"audio,text",0.000033,music/Swirling Ship - Shaal Region - Short Ver...,"Moods: Carefree, Love, Peaceful, Playful, Hope..."
3,Good Times,Kyle Preston,"Uplifting, Carefree, Love, Peaceful, Serious, ...","Time-Lapse, Documentary, Road Trip, Lifestyle,...",Cinematic,Piano,,A warm and nostalgic solo piano track with a c...,0.781314,0.788826,0.778094,"audio,text",0.000032,music/Kyle Preston - Good Times.mp3,"Moods: Uplifting, Carefree, Love, Peaceful, Se..."
4,Synth and Whistle,Artlist Musical Logos,"Peaceful, Hopeful",Intros & Logos,"Cinematic, Pop","Piano, Whistle, Pads",,A peaceful and tranquil track featuring an Iri...,0.738893,0.701042,0.755115,"audio,text",0.000027,music/Artlist Musical Logos - Synth and Whistl...,"Moods: Peaceful, Hopeful. Video Themes: Intros..."
5,Modern Club Party,Difourks,"dirty, energetic, energy, extreme, positive","blogging, club, dancing, game, internet, party...","dance, electro, electronic, house, pop",bass,"loop, looped, modern",A powerful and upbeat techno track with a groo...,0.862010,0.540032,1.000000,"audio,text",0.000027,music/Difourks - Modern Club Party.wav,"Moods: dirty, energetic, energy, extreme, posi..."
6,Acoustic Guitar Romantic Bright,VICTORMUSIC,"beautiful, bright, calm, comforting, earthy, g...","advertising, commercial, documentary, family, ...","folk, indie","acoustic guitar, fingerstyle",,A calm and positive acoustic guitar instrument...,0.827788,0.819341,0.831408,"audio,text",0.000019,music/VICTORMUSIC - Acoustic Guitar Romantic B...,"Moods: beautiful, bright, calm, comforting, ea..."
7,Health,fatbunny,"bright, future, innovation, motivational, success","advertise, advertising, business, commercial, ...","ambient, corporate, orchestral",,modern,"A calm, uplifting, and inspiring background tr...",0.808904,0.711032,0.850849,"audio,text",0.000016,music/fatbunny - Health.wav,"Moods: bright, future, innovation, motivationa..."
8,Forest Landspace,Artlist Musical Logos,"Peaceful, Serious","Technology, Lifestyle, Commercial, Vlog, Intro...",Cinematic,"Piano, Pads",,A soft and ambient background track featuring ...,0.756205,0.426104,0.897678,"audio,text",0.000015,music/Artlist Musical Logos - Forest Landspace...,"Moods: Peaceful, Serious. Video Themes: Techno..."
9,Corporate Loop Presentation,Positive_Sound,"atmospheric, calm, dream, flowing, freedom, ho...","advertising, beauty, business, commercials, li...",corporate,,"loop, loopable","A positive, inspiring, and uplifting backgroun...",0.744035,0.711549,0.757957,"audio,text",0.000014,music/Positive_Sound - Corporate Loop Presenta...,"Moods: atmospheric, calm, dream, flowing, free..."


In [5]:
print(results["choice"])

openai


In [6]:
print(results["retrieval_context"])

1. Ziv Moran - Promised - Short Version B
Information: Moods: Uplifting, Powerful, Hopeful, Groovy, Exciting. Video Themes: Road Trip, Lifestyle, Urban, Medical, Landscape, Travel. Instruments: Acoustic Guitar, Piano, Acoustic Drums, Bells, Backing Vocals. Genres: Acoustic, Pop, Folk. Other tags: . Description: A driving, upbeat, acoustic guitar-based instrumental with a driving beat and a sense of movement and progress. The mood is positive and uplifting, with a sense of accomplishment and satisfaction. The main instruments are acoustic guitar, piano, and drums. The genre/style is pop, folk, and instrumental. Suitable uses for this track include background music for corporate videos, presentations, and commercials, as well as in film and television soundtracks.




In [7]:
print(results["explanation"])

The user's need for a tool to clear browsing history does not relate to music preferences, making it challenging to establish a connection with the recommended song. "Promised - Short Version B" by Ziv Moran is an uplifting and powerful acoustic track, characterized by its driving beat and sense of progress. While the song's positive and hopeful mood might metaphorically align with a sense of digital 'cleanliness' or 'fresh start,' it doesn't directly address the user's specific request. Therefore, no strong musical connection can be reasonably inferred.


In [8]:
print(results["explanation_prompt"])

You are a professional music recommendation assistant.
        Based on the following user need and the details of recommended songs, please generate a natural, clear, and engaging explanation.
        
        User needs: I’m looking for a tool to clear my browsing history.

        Recommended songs:
        1. Ziv Moran - Promised - Short Version B
Information: Moods: Uplifting, Powerful, Hopeful, Groovy, Exciting. Video Themes: Road Trip, Lifestyle, Urban, Medical, Landscape, Travel. Instruments: Acoustic Guitar, Piano, Acoustic Drums, Bells, Backing Vocals. Genres: Acoustic, Pop, Folk. Other tags: . Description: A driving, upbeat, acoustic guitar-based instrumental with a driving beat and a sense of movement and progress. The mood is positive and uplifting, with a sense of accomplishment and satisfaction. The main instruments are acoustic guitar, piano, and drums. The genre/style is pop, folk, and instrumental. Suitable uses for this track include background music for corporate vi

In [9]:
judge_llm = CustomClaudeSonnet()

In [10]:
query = query 
generated_output = results["explanation"]
context_docs = [results["retrieval_context"]]

results = evaluate_all_metrics_with_usefulness(query, generated_output, context_docs, model = judge_llm)
output = {}
for test_result in results.test_results:
    for metric_result in test_result.metrics_data:
        output[metric_result.name] = {
            "score": metric_result.score,
            "reason": getattr(metric_result, "reason", "N/A"),
            "pass": metric_result.success
        }
output

✨ You're running DeepEval's latest Faithfulness Metric! (using Claude-3.5-Sonnet, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using Claude-3.5-Sonnet, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Usefulness (GEval) Metric! (using Claude-3.5-Sonnet, strict=False, 
async_mode=True)...

Evaluating 1 test case(s) in parallel: |██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|100% (1/1) [Time Taken: 00:24, 24.79s/test case]



Metrics Summary

  - ✅ Faithfulness (score: 1.0, threshold: 0.7, strict: False, evaluation model: Claude-3.5-Sonnet, reason: The score is 1.00 because the response shows perfect faithfulness! There are no contradictions between the actual output and the retrieval context, indicating the model did an excellent job staying true to the source material., error: None)
  - ❌ Answer Relevancy (score: 0.6666666666666666, threshold: 0.7, strict: False, evaluation model: Claude-3.5-Sonnet, reason: The score is 0.67 because while the response did address browser history clearing tools, it contained unnecessary and off-topic information about music tracks and song characteristics that had nothing to do with the user's request for browser history clearing assistance. The score still remains above average since the core request was addressed, but the irrelevant musical details prevented it from scoring higher., error: None)
  - ✅ Usefulness (GEval) (score: 0.9, threshold: 0.7, strict: False, evalu

✓ Tests finished 🎉! Run 'deepeval login' to save and analyze evaluation results on Confident AI.
 
✨👀 Looking for a place for your LLM test data to live 🏡❤️ ? Use Confident AI to get & share testing reports, 
experiment with models/prompts, and catch regressions for your LLM system. Just run 'deepeval login' in the CLI.

{'Faithfulness': {'score': 1.0,
  'reason': 'The score is 1.00 because the response shows perfect faithfulness! There are no contradictions between the actual output and the retrieval context, indicating the model did an excellent job staying true to the source material.',
  'pass': True},
 'Answer Relevancy': {'score': 0.6666666666666666,
  'reason': "The score is 0.67 because while the response did address browser history clearing tools, it contained unnecessary and off-topic information about music tracks and song characteristics that had nothing to do with the user's request for browser history clearing assistance. The score still remains above average since the core request was addressed, but the irrelevant musical details prevented it from scoring higher.",
  'pass': False},
 'Usefulness (GEval)': {'score': 0.9,
  'reason': "The explanation acknowledges the mismatch between the browsing history request and music recommendation, shows honesty by clearly stating 'no strong musical 